In [ ]:
from google.colab import files

uploaded = files.upload()

Saving train.csv to train (1).csv


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving test.csv to test (1).csv


In [ ]:
%matplotlib inline
# Pandas is a nice utilitiy that enables some easy data manipulation, especially from a csv
import pandas as pd
# Numpy lets us work with arrays
import numpy as np
import random
# Sklearn provides various modules with a common API
from sklearn import svm, tree, neighbors, neural_network
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

In [ ]:
# Read in our csv files downloaded from Kaggle
train_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

In [ ]:
# We can use .head() to get just the first few rows of the data.
train_data.head()
# Note how the notebook nicely displays the output for us

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [ ]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [ ]:
print("Shape:", train_data.shape)
print("Columns:", train_data.columns.tolist())

Shape: (891, 12)
Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


In [ ]:
print(train_data.isna().sum())

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [ ]:
train_data["FamilySize"] = (
    train_data["SibSp"]
    + train_data["Parch"]
    + 1
)

test_data["FamilySize"] = (
    test_data["SibSp"]
    + test_data["Parch"]
    + 1
)
train_data["IsAlone"] = (
    train_data["FamilySize"] == 1
).astype(int)

test_data["IsAlone"] = (
    test_data["FamilySize"] == 1
).astype(int)

In [ ]:
train_data["FarePerPerson"] = (
    train_data["Fare"]
    / train_data["FamilySize"]
)

test_data["FarePerPerson"] = (
    test_data["Fare"]
    / test_data["FamilySize"]
)
# Extract passenger title from Name
train_data["Title"] = train_data["Name"].str.extract(
    r",\s*([^.]*)\.",
    expand=False
)

test_data["Title"] = test_data["Name"].str.extract(
    r",\s*([^.]*)\.",
    expand=False
)


# Keep the four common titles and combine everything else
common_titles = [
    "Mr",
    "Mrs",
    "Miss",
    "Master"
]

train_data["Title"] = train_data["Title"].where(
    train_data["Title"].isin(common_titles),
    "Rare"
)

test_data["Title"] = test_data["Title"].where(
    test_data["Title"].isin(common_titles),
    "Rare"
)


# Convert titles to numbers
title_map = {
    "Mr": 0,
    "Mrs": 1,
    "Miss": 2,
    "Master": 3,
    "Rare": 4
}

train_data["Title"] = train_data["Title"].map(
    title_map
)

test_data["Title"] = test_data["Title"].map(
    title_map
)

In [ ]:
train_data[
    [
        "SibSp",
        "Parch",
        "FamilySize",
        "IsAlone",
        "Fare",
        "FarePerPerson",
        "Title"
    ]
].head(10)

,SibSp,Parch,FamilySize,IsAlone,Fare,FarePerPerson
0,1,0,2,0,7.2500,3.62500
1,1,0,2,0,71.2833,35.64165
2,0,0,1,1,7.9250,7.92500
3,1,0,2,0,53.1000,26.55000
4,0,0,1,1,8.0500,8.05000
5,0,0,1,1,8.4583,8.45830
6,0,0,1,1,51.8625,51.86250
7,3,1,5,0,21.0750,4.21500
8,0,2,3,0,11.1333,3.71110
9,1,0,2,0,30.0708,15.03540


In [ ]:
random.seed(100)
np.random.seed(100)
train_data.drop(
    columns=["Name", "Ticket", "Cabin"],
    inplace=True
)

test_data.drop(
    columns=["Name", "Ticket", "Cabin"],
    inplace=True
)
train_data.set_index(
    keys=["PassengerId"],
    drop=True,
    inplace=True
)

test_data.set_index(
    keys=["PassengerId"],
    drop=True,
    inplace=True
)

In [ ]:
train_nan_map = {
    "Age": train_data["Age"].median(),
    "Fare": train_data["Fare"].median(),
    "Embarked": train_data["Embarked"].mode()[0]
}

test_nan_map = {
    "Age": test_data["Age"].median(),
    "Fare": test_data["Fare"].median(),
    "Embarked": test_data["Embarked"].mode()[0]
}

In [ ]:
train_data.fillna(
    value=train_nan_map,
    inplace=True
)

test_data.fillna(
    value=test_nan_map,
    inplace=True
)

In [ ]:
train_data["FarePerPerson"] = (
    train_data["Fare"]
    / train_data["FamilySize"]
)

test_data["FarePerPerson"] = (
    test_data["Fare"]
    / test_data["FamilySize"]
)

In [ ]:
columns_map = {
    "Embarked": {
        "C": 0,
        "Q": 1,
        "S": 2
    },
    "Sex": {
        "male": 0,
        "female": 1
    }
}

In [ ]:
train_data.replace(
    columns_map,
    inplace=True
)

test_data.replace(
    columns_map,
    inplace=True
)

/tmp/ipykernel_769/3278339506.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_data.replace(
/tmp/ipykernel_769/3278339506.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test_data.replace(


In [ ]:
train_data["Sex"] = train_data["Sex"].astype(int)
train_data["Embarked"] = train_data["Embarked"].astype(int)

test_data["Sex"] = test_data["Sex"].astype(int)
test_data["Embarked"] = test_data["Embarked"].astype(int)

In [ ]:
print(
    "Training missing values:",
    train_data.isna().sum().sum()
)

print(
    "Test missing values:",
    test_data.isna().sum().sum()
)

Training missing values: 0
Test missing values: 0


In [ ]:
train_data.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,FamilySize,IsAlone,FarePerPerson
PassengerId,,,,,,,,,,,
1,0,3,0,22.0,1,0,7.2500,2,2,0,3.62500
2,1,1,1,38.0,1,0,71.2833,0,2,0,35.64165
3,1,3,1,26.0,0,0,7.9250,2,1,1,7.92500
4,1,1,1,35.0,1,0,53.1000,2,2,0,26.55000
5,0,3,0,35.0,0,0,8.0500,2,1,1,8.05000


In [ ]:
X = train_data.loc[
    :,
    train_data.columns != "Survived"
]

y = train_data.loc[:, "Survived"]

In [ ]:
print(X.head())
print(y.head())

             Pclass  Sex   Age  SibSp  Parch     Fare  Embarked  FamilySize  \
PassengerId                                                                   
1                 3    0  22.0      1      0   7.2500         2           2   
2                 1    1  38.0      1      0  71.2833         0           2   
3                 3    1  26.0      0      0   7.9250         2           1   
4                 1    1  35.0      1      0  53.1000         2           2   
5                 3    0  35.0      0      0   8.0500         2           1   

             IsAlone  FarePerPerson  
PassengerId                          
1                  0        3.62500  
2                  0       35.64165  
3                  1        7.92500  
4                  0       26.55000  
5                  1        8.05000  
PassengerId
1    0
2    1
3    1
4    1
5    0
Name: Survived, dtype: int64


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=10,
    stratify=y
)

In [ ]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (596, 10)
X_test shape: (295, 10)
y_train shape: (596,)
y_test shape: (295,)


In [ ]:
print("Training survival rate:", y_train.mean())
print("Validation survival rate:", y_test.mean())

Training survival rate: 0.38422818791946306
Validation survival rate: 0.38305084745762713


In [ ]:
random.seed(100)
np.random.seed(100)

tree_clf = tree.DecisionTreeClassifier(
    max_depth=5,
    random_state=100
)

tree_clf.fit(
    X_train.values,
    y_train.values
)

DecisionTreeClassifier(max_depth=5, random_state=100)

In [ ]:
train_accuracy = tree_clf.score(
    X_train.values,
    y_train.values
)

test_accuracy = tree_clf.score(
    X_test.values,
    y_test.values
)

print("Training accuracy:", train_accuracy)
print("Validation accuracy:", test_accuracy)

Training accuracy: 0.8674496644295302
Validation accuracy: 0.8067796610169492


In [ ]:
y_pred = tree_clf.predict(X_test.values)
y_truth = y_test.values
cm = confusion_matrix(
    y_truth,
    y_pred,
    labels=[0, 1]
)

print("Confusion Matrix")
print(cm)

Confusion Matrix
[[153  29]
 [ 28  85]]


In [ ]:
tn, fp, fn, tp = cm.ravel()

print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)
fpr = fp / (fp + tn)
fnr = fn / (fn + tp)

print("False Positive Rate:", fpr)
print("False Negative Rate:", fnr)

True Negatives: 153
False Positives: 29
False Negatives: 28
True Positives: 85
False Positive Rate: 0.15934065934065933
False Negative Rate: 0.24778761061946902


In [ ]:
predictions = tree_clf.predict(test_data.values)
print(type(predictions))
print("Number of predictions:", len(predictions))
pred_df = pd.DataFrame(
    predictions,
    index=test_data.index,
    columns=["Survived"]
)

<class 'numpy.ndarray'>
Number of predictions: 418


In [ ]:
pred_df.head()

,Survived
PassengerId,
892,0
893,0
894,0
895,0
896,1


In [ ]:
pred_df.to_csv(
    "decision_tree_predictions.csv",
    header=True,
    sep=","
)

In [ ]:
saved_predictions = pd.read_csv(
    "decision_tree_predictions.csv"
)

saved_predictions.head()

,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


In [ ]:
# Santoshi Yadagiri — Support Vector Machine

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
from sklearn.base import clone

svm_clf = make_pipeline(
    StandardScaler(),
    SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale"
    )
)

svm_clf.fit(
    X_train.values,
    y_train.values
)

Pipeline(steps=[('standardscaler', StandardScaler()), ('svc', SVC())])

In [ ]:
svm_train_accuracy = svm_clf.score(
    X_train.values,
    y_train.values
)

svm_test_accuracy = svm_clf.score(
    X_test.values,
    y_test.values
)

print("Training accuracy:", svm_train_accuracy)
print("Validation accuracy:", svm_test_accuracy)

Training accuracy: 0.8406040268456376
Validation accuracy: 0.8169491525423729


In [ ]:
svm_y_pred = svm_clf.predict(X_test.values)
svm_y_truth = y_test.values

svm_cm = confusion_matrix(
    svm_y_truth,
    svm_y_pred,
    labels=[0, 1]
)

print("Confusion Matrix")
print(svm_cm)

Confusion Matrix
[[162  20]
 [ 34  79]]


In [ ]:
svm_tn, svm_fp, svm_fn, svm_tp = svm_cm.ravel()

print("True Negatives:", svm_tn)
print("False Positives:", svm_fp)
print("False Negatives:", svm_fn)
print("True Positives:", svm_tp)

svm_fpr = svm_fp / (svm_fp + svm_tn)
svm_fnr = svm_fn / (svm_fn + svm_tp)

print("False Positive Rate:", svm_fpr)
print("False Negative Rate:", svm_fnr)

True Negatives: 162
False Positives: 20
False Negatives: 34
True Positives: 79
False Positive Rate: 0.10989010989010989
False Negative Rate: 0.3008849557522124


In [ ]:
## Validation Results

## Using the shared training/validation split, my SVM achieved:

##- Training accuracy: 84.06%
##- Validation accuracy: 81.69%
##- True negatives: 162
##- False positives: 20
##- False negatives: 34
##- True positives: 79
##- False positive rate: 10.99%
##- False negative rate: 30.09%

## A false positive means the model predicted survival for a passenger who did not survive. A false negative means the model predicted nonsurvival for a passenger who survived.

## These results are from the 295-passenger validation portion of train.csv, not the unlabeled Kaggle test.csv. I used the fixed SVM settings above without a hyperparameter search.

## I preserved the team's existing preprocessing for consistency. One limitation is that missing-value imputation was performed before the training/validation split, so validation feature statistics influenced the imputation values.

In [ ]:
svm_final = clone(svm_clf)

svm_final.fit(
    X.values,
    y.values
)

svm_predictions = svm_final.predict(
    test_data.loc[:, X.columns].values
)

print(type(svm_predictions))
print("Number of predictions:", len(svm_predictions))

svm_pred_df = pd.DataFrame(
    svm_predictions.astype(int),
    index=test_data.index,
    columns=["Survived"]
)

svm_pred_df.index.name = "PassengerId"

svm_pred_df.head()

<class 'numpy.ndarray'>
Number of predictions: 418


,Survived
PassengerId,
892,0
893,1
894,0
895,0
896,1


In [ ]:
svm_pred_df.to_csv(
    "santoshi_svm_predictions.csv",
    header=True,
    sep=","
)

In [ ]:
svm_saved_predictions = pd.read_csv(
    "santoshi_svm_predictions.csv"
)

assert svm_saved_predictions.shape == (418, 2)
assert svm_saved_predictions.columns.tolist() == [
    "PassengerId", "Survived"
]
assert svm_saved_predictions["PassengerId"].tolist() == (
    test_data.index.tolist()
)
assert svm_saved_predictions["Survived"].isin([0, 1]).all()
assert not svm_saved_predictions.isna().any().any()

svm_saved_predictions.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [ ]:
# Andrew Freeman - Random Forest Classifier

In [ ]:
from pathlib import Path
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.base import clone

# In Colab, use existing files or prompt you to upload the missing ones.
# Upload the original Titanic train.csv and test.csv.
def load_csv(filename):
    path = Path(filename)

    if path.exists():
        return pd.read_csv(path)

    from google.colab import files
    import io

    print(f"Please upload {filename}")
    uploaded = files.upload()

    if filename not in uploaded:
        raise ValueError(f"Expected an uploaded file named {filename}.")

    return pd.read_csv(io.BytesIO(uploaded[filename]))


rf_train = load_csv("train.csv")
rf_test = load_csv("test.csv")

# Create features separately from the original data.
# These calculations use no survival labels or dataset-wide statistics.
def create_features(data):
    features = data[
        ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex", "Embarked"]
    ].copy()

    features["FamilySize"] = features["SibSp"] + features["Parch"] + 1
    features["IsAlone"] = (features["FamilySize"] == 1).astype(int)

    # This is a fare-to-family-size ratio, not a verified individual fare.
    # Missing ratios will be filled by the pipeline.
    features["FarePerPerson"] = (
        features["Fare"] / features["FamilySize"]
    )
    return features


rf_X = create_features(rf_train)
rf_y = rf_train["Survived"]
rf_test_features = create_features(rf_test)

numeric = [
    "Pclass", "Age", "SibSp", "Parch", "Fare",
    "FamilySize", "IsAlone", "FarePerPerson"
]
categorical = ["Sex", "Embarked"]

# Learn missing-value replacements only from the training portion.
# One-hot encoding avoids imposing an order on embarkation categories.
preprocessor = ColumnTransformer([
    ("numeric", SimpleImputer(strategy="median"), numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
])

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,     # Combine 200 trees.
        max_depth=6,          # Limit tree complexity.
        min_samples_leaf=3,   # Avoid very small leaves.
        random_state=42,
        n_jobs=-1
    ))
])

# Match the new notebook's split settings.
# Stratification preserves similar survival proportions in each portion.
rf_X_train, rf_X_valid, rf_y_train, rf_y_valid = train_test_split(
    rf_X,
    rf_y,
    test_size=0.33,
    random_state=10,
    stratify=rf_y
)

rf_model.fit(rf_X_train, rf_y_train)

print(f"Training accuracy: {rf_model.score(rf_X_train, rf_y_train):.2%}")
print(f"Validation accuracy: {rf_model.score(rf_X_valid, rf_y_valid):.2%}")

valid_predictions = rf_model.predict(rf_X_valid)
rf_cm = confusion_matrix(rf_y_valid, valid_predictions, labels=[0, 1])
tn, fp, fn, tp = rf_cm.ravel()

print("\nConfusion matrix (rows = actual, columns = predicted):")
print(rf_cm)

print("\nTrue negatives:", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives:", tp)

print(f"\nFalse positive rate: {fp / (fp + tn):.2%}")
print(f"False negative rate: {fn / (fn + tp):.2%}")

Training accuracy: 88.76%
Validation accuracy: 83.39%

Confusion matrix (rows = actual, columns = predicted):
[[163  19]
 [ 30  83]]

True negatives: 163
False positives: 19
False negatives: 30
True positives: 83

False positive rate: 10.44%
False negative rate: 26.55%


In [ ]:
# Retrain a fresh copy on all labeled passengers.
# The validation model above stays unchanged.
rf_final = clone(rf_model)
rf_final.fit(rf_X, rf_y)

test_predictions = rf_final.predict(rf_test_features)

rf_submission = pd.DataFrame({
    "PassengerId": rf_test["PassengerId"],
    "Survived": test_predictions.astype(int)
})

# Verify the required submission format.
assert rf_submission.shape == (418, 2)
assert rf_submission.columns.tolist() == ["PassengerId", "Survived"]
assert rf_submission["PassengerId"].is_unique
assert rf_submission["Survived"].isin([0, 1]).all()
assert not rf_submission.isna().any().any()

output_file = "random_forest_predictions.csv"
rf_submission.to_csv(output_file, index=False)

print(f"Saved {output_file}")
print(rf_submission.head())

# Download the finished file when running in Google Colab.
try:
    from google.colab import files
except ImportError:
    print(f"File location: {Path(output_file).resolve()}")
else:
    files.download(output_file)

In [ ]:
# Mannat Kaur - Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

logreg_clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=100)
)

In [ ]:
logreg_clf.fit(X_train, y_train)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(max_iter=1000, random_state=100))])

In [ ]:
train_accuracy = logreg_clf.score(X_train, y_train)
test_accuracy = logreg_clf.score(X_test, y_test)

print("Training Accuracy:", train_accuracy)
print("Validation Accuracy:", test_accuracy)

Training Accuracy: 0.802013422818792
Validation Accuracy: 0.7898305084745763


In [ ]:
y_pred = logreg_clf.predict(X_test)
print("First 20 predictions:")
print(y_pred[:20])

First 20 predictions:
[0 0 1 1 0 1 1 0 0 0 0 1 0 0 0 1 0 1 0 0]


In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[153  29]
 [ 33  80]]


In [ ]:
tn, fp, fn, tp = cm.ravel()
print("True Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)

false_positive_rate = fp / (fp + tn)
false_negative_rate = fn / (fn + tp)

print("False Positive Rate:", false_positive_rate)
print("False Negative Rate:", false_negative_rate)

True Negatives: 153
False Positives: 29
False Negatives: 33
True Positives: 80
False Positive Rate: 0.15934065934065933
False Negative Rate: 0.2920353982300885


In [ ]:
c_values = [0.01, 0.1, 1, 10, 100]

for c in c_values:
  model = make_pipeline(
      StandardScaler(),
      LogisticRegression(
          C=c,
          max_iter=1000,
          random_state=100
      )
  )

  model.fit(X_train, y_train)

  train_acc = model.score(X_train, y_train)
  val_acc = model.score(X_test, y_test)

  print(
      "C =", c,
      "| Training Accuracy =", round(train_acc, 4),
      "| Validation Accuracy =", round(val_acc, 4)
  )

C = 0.01 | Training Accuracy = 0.8037 | Validation Accuracy = 0.7627
C = 0.1 | Training Accuracy = 0.802 | Validation Accuracy = 0.7932
C = 1 | Training Accuracy = 0.802 | Validation Accuracy = 0.7898
C = 10 | Training Accuracy = 0.802 | Validation Accuracy = 0.7831
C = 100 | Training Accuracy = 0.802 | Validation Accuracy = 0.7831


In [ ]:
logreg_final = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=0.1,
        max_iter=1000,
        random_state=100
    )
)

logreg_final.fit(X_train, y_train)
print("Training Accuracy:", logreg_final.score(X_train, y_train))
print("Validation Accuracy:", logreg_final.score(X_test, y_test))

Training Accuracy: 0.802013422818792
Validation Accuracy: 0.7932203389830509


In [ ]:
final_probabilities = logreg_final.predict_proba(X_test)[:, 1]
final_y_pred = (final_probabilities >= 0.40).astype(int)

final_cm = confusion_matrix(y_test, final_y_pred)

tn, fp, fn, tp = final_cm.ravel()

false_positive_rate = fp / (fp + tn)
false_negative_rate = fn / (fn + tp)

print("Confusion Matrix:")
print(final_cm)
print("False Positive Rate:", false_positive_rate)
print("False Negative Rate:", false_negative_rate)

Confusion Matrix:
[[143  39]
 [ 27  86]]
False Positive Rate: 0.21428571428571427
False Negative Rate: 0.23893805309734514


In [ ]:
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

survival_probabilities = logreg_final.predict_proba(X_test)[:, 1]

for threshold in thresholds:
    threshold_predictions = (survival_probabilities >= threshold).astype(int)

    cm = confusion_matrix(y_test, threshold_predictions)
    tn, fp, fn, tp = cm.ravel()

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    accuracy = (tp + tn) / len(y_test)

    print(
        "Threshold =", threshold,
        "| Accuracy =", round(accuracy, 4),
        "| FPR =", round(fpr, 4),
        "| FNR =", round(fnr, 4)
    )

Threshold = 0.3 | Accuracy = 0.7559 | FPR = 0.3132 | FNR = 0.1327
Threshold = 0.35 | Accuracy = 0.7729 | FPR = 0.2637 | FNR = 0.1681
Threshold = 0.4 | Accuracy = 0.7763 | FPR = 0.2143 | FNR = 0.2389
Threshold = 0.45 | Accuracy = 0.7831 | FPR = 0.1813 | FNR = 0.2743
Threshold = 0.5 | Accuracy = 0.7932 | FPR = 0.1538 | FNR = 0.292
Threshold = 0.55 | Accuracy = 0.8 | FPR = 0.1099 | FNR = 0.3451
Threshold = 0.6 | Accuracy = 0.7797 | FPR = 0.0769 | FNR = 0.4513
Threshold = 0.65 | Accuracy = 0.7559 | FPR = 0.0659 | FNR = 0.531
Threshold = 0.7 | Accuracy = 0.739 | FPR = 0.0549 | FNR = 0.5929


In [ ]:
final_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        C=0.1,
        max_iter=1000,
        random_state=100
    )
)

final_model.fit(X, y)

Pipeline(steps=[('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(C=0.1, max_iter=1000, random_state=100))])

In [ ]:
X_kaggle_test = test_data[X.columns]
kaggle_probabilities = final_model.predict_proba(X_kaggle_test)[:, 1]

kaggle_predictions = (
    kaggle_probabilities >= 0.40
).astype(int)

print("Number of predictions:", len(kaggle_predictions))
print("First 20 predictions:", kaggle_predictions[:20])

Number of predictions: 418
First 20 predictions: [0 1 0 0 1 0 1 0 1 0 0 0 1 0 1 1 0 0 1 1]


In [ ]:
submission = pd.DataFrame({
    "PassengerId": test_data.index,
    "Survived": kaggle_predictions.astype(int)
})

submission.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [ ]:
submission.to_csv("logistic_regression_submission.csv", index=False)

print("Submission file created!")

Submission file created!


In [ ]:
from google.colab import files

files.download("logistic_regression_submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
## Suhanraj Thirumal, Gaussian Naive Bayes

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix
from google.colab import files

# Upload train.csv and test.csv together
uploaded = files.upload()

# File loading
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

# Create the same engineered features used by the group
train_data["FamilySize"] = (
    train_data["SibSp"]
    + train_data["Parch"]
    + 1
)

test_data["FamilySize"] = (
    test_data["SibSp"]
    + test_data["Parch"]
    + 1
)

train_data["IsAlone"] = (
    train_data["FamilySize"] == 1
).astype(int)

test_data["IsAlone"] = (
    test_data["FamilySize"] == 1
).astype(int)

# Fill missing information
train_data["Age"] = train_data["Age"].fillna(
    train_data["Age"].median()
)

test_data["Age"] = test_data["Age"].fillna(
    test_data["Age"].median()
)

train_data["Fare"] = train_data["Fare"].fillna(
    train_data["Fare"].median()
)

test_data["Fare"] = test_data["Fare"].fillna(
    test_data["Fare"].median()
)

train_data["Embarked"] = train_data["Embarked"].fillna(
    train_data["Embarked"].mode()[0]
)

test_data["Embarked"] = test_data["Embarked"].fillna(
    test_data["Embarked"].mode()[0]
)

# Calculate fare per person after missing fares are filled
train_data["FarePerPerson"] = (
    train_data["Fare"]
    / train_data["FamilySize"]
)

test_data["FarePerPerson"] = (
    test_data["Fare"]
    / test_data["FamilySize"]
)

# Change text categories into numbers
sex_map = {
    "male": 0,
    "female": 1
}

embarked_map = {
    "C": 0,
    "Q": 1,
    "S": 2
}

train_data["Sex"] = train_data["Sex"].map(sex_map)
test_data["Sex"] = test_data["Sex"].map(sex_map)

train_data["Embarked"] = train_data["Embarked"].map(embarked_map)
test_data["Embarked"] = test_data["Embarked"].map(embarked_map)

# Passenger information the model ingest
features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "FamilySize",
    "IsAlone",
    "FarePerPerson"
]

# Create the model inputs
X = train_data[features].copy()
X_kaggle_test = test_data[features].copy()

# The answer the model learns to predict
y = train_data["Survived"]

# Create training and validation groups
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=10,
    stratify=y
)

# create and train the Gaussian Naive Bayes model
n_bayes_model = GaussianNB()
n_bayes_model.fit(X_train, y_train)

# Make training and validation predictions
training_predicts = n_bayes_model.predict(X_train)
val_predicts = n_bayes_model.predict(X_val)

# Calculate accuracy
training_accuracy = accuracy_score(
    y_train,
    training_predicts
)

val_accuracy = accuracy_score(
    y_val,
    val_predicts
)

print(
    "Training Accuracy:",
    round(training_accuracy, 4)
)

print(
    "Validation Accuracy:",
    round(val_accuracy, 4)
)

# make the confusion matrix
matrix = confusion_matrix(
    y_val,
    val_predicts,
    labels=[0, 1])

print("Confusion Matrix:")
print(matrix)

# separate the confusion matrix results
true_negative, false_positive, false_negative, true_positive = (
    matrix.ravel())

print("True Negatives:", true_negative)
print("False Positives:", false_positive)
print("False Negatives:", false_negative)
print("True Positives:", true_positive)

# calculate false-positive and false-negative rates
false_positive_rate = false_positive / (
    false_positive + true_negative)

false_negative_rate = false_negative / (
    false_negative + true_positive)

print(
    "False Positive Rate:",
    round(false_positive_rate, 4))

print(
    "False Negative Rate:",
    round(false_negative_rate, 4))

# train the model again using all the training data
n_bayes_model.fit(X, y)

# predict the answers for test.csv
test_predicts = n_bayes_model.predict(
    X_kaggle_test
)

# make the final prediction file
submission = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": test_predicts.astype(int)})

submission.to_csv(
    "gaussian_naive_bayes_predictions.csv",
    index=False)

print("Prediction file created!")
print("Number of predictions:", len(submission))
print(submission.head())

# download the final prediction file
files.download(
    "gaussian_naive_bayes_predictions.csv")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix


def calculate_error_rates(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    accuracy = (
        (tn + tp)
        / (tn + fp + fn + tp)
    )

    return {
        "Accuracy": accuracy,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "FPR": fpr,
        "FNR": fnr
    }

In [ ]:
# Decision Tree
tree_validation_predictions = tree_clf.predict(
    X_test.values
)

# Support Vector Machine
svm_validation_predictions = svm_clf.predict(
    X_test.values
)

# Random Forest
rf_validation_predictions = rf_model.predict(
    rf_X_valid
)

# Logistic Regression
# Uses the model's normal 0.50 decision boundary
logreg_validation_predictions = logreg_final.predict(
    X_test
)

# Gaussian Naive Bayes
nb_validation_predictions = nb_model.predict(
    nb_X_val
)



NameError: name 'nb_model' is not defined

In [ ]:
pareto_results = pd.DataFrame([
    {
        "Model": "Decision Tree",
        **tree_metrics
    },
    {
        "Model": "SVM",
        **svm_metrics
    },
    {
        "Model": "Random Forest",
        **rf_metrics
    },
    {
        "Model": "Logistic Regression",
        **logreg_metrics
    },
    {
        "Model": "Gaussian Naive Bayes",
        **nb_metrics
    }
])

pareto_results[
    [
        "Model",
        "Accuracy",
        "FPR",
        "FNR"
    ]
].sort_values("FPR")
def is_dominated(candidate_index, results):
    candidate = results.loc[candidate_index]

    for other_index, other in results.iterrows():
        if other_index == candidate_index:
            continue

        no_worse = (
            other["FPR"] <= candidate["FPR"]
            and other["FNR"] <= candidate["FNR"]
        )

        strictly_better = (
            other["FPR"] < candidate["FPR"]
            or other["FNR"] < candidate["FNR"]
        )

        if no_worse and strictly_better:
            return True

    return False

In [ ]:
pareto_results["Dominated"] = [
    is_dominated(i, pareto_results)
    for i in pareto_results.index
]

pareto_results["ParetoOptimal"] = (
    ~pareto_results["Dominated"]
)

pareto_results[
    [
        "Model",
        "Accuracy",
        "FPR",
        "FNR",
        "Dominated",
        "ParetoOptimal"
    ]
].sort_values("FPR")